# Piper-Neo Studio · Notebook oficial de Kaggle

**Repositorio oficial, configurado automáticamente:** [ThowiLabs/Piper-Neo](https://github.com/ThowiLabs/Piper-Neo) · rama `feature/resilient-finetune-studio`.

No tienes que proporcionar una URL de repositorio, subir un ZIP ni elegir una fuente de instalación. Este cuaderno reclona el código oficial en cada ejecución, sin tocar /kaggle/working/piper_finetune, e instala dependencias fijadas, verifica GPU NVIDIA y abre Gradio con preparación de dataset, fine-tune, recuperación HF, backups e inferencia.

**Antes de ejecutar:** activa en Kaggle **Settings → Accelerator → GPU** y **Internet → ON**. La instalación inicial requiere Internet y varios GB de espacio, además de lo necesario para tus datasets y checkpoints.

**Seguridad:** los secretos HF van en Kaggle Secrets, no dentro del notebook. Un enlace público Gradio sin contraseña es accesible para cualquiera que conozca la URL. Si vas a usar tokens, configura `PIPER_STUDIO_AUTH_USER` y `PIPER_STUDIO_AUTH_PASSWORD` en Kaggle Secrets.


In [ ]:
# 1. Origen oficial fijo y raíz de datos SEPARADA del clon de código.
import os
REPO_URL = "https://github.com/ThowiLabs/Piper-Neo.git"
BRANCH = "feature/resilient-finetune-studio"
ROOT = "/kaggle/working/Piper-Neo"
DATA_ROOT = "/kaggle/working/piper_finetune"
os.environ["PIPER_FINETUNE_ROOT"] = DATA_ROOT
os.environ["GRADIO_TEMP_DIR"] = DATA_ROOT + "/gradio_tmp"
print("Repositorio oficial:", REPO_URL)
print("Rama Studio:", BRANCH)
print("Datos y checkpoints fuera del clon:", DATA_ROOT)


In [ ]:
# 2. Reinstalación fresca del CÓDIGO, sin borrar datos ni entrenamientos.
# SOLO se ejecuta en Kaggle cuando ejecutes este cuaderno; nunca en tu PC.
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

project_dir = Path(ROOT)
data_dir = Path(DATA_ROOT)
root_parent = project_dir.parent
if project_dir.is_symlink():
    raise RuntimeError("Piper-Neo es un enlace simbólico: no se borra automáticamente.")
if project_dir.resolve() == data_dir.resolve() or project_dir.resolve() in data_dir.resolve().parents:
    raise RuntimeError("Los datos persistentes están dentro del clon. Reubícalos antes de reinstalar.")

if project_dir.exists():
    if not project_dir.is_dir() or not (project_dir / ".git").is_dir():
        raise RuntimeError("Piper-Neo existe pero no es un clon Git reconocido. No se elimina.")
    configured_origin = subprocess.check_output(
        ["git", "-C", ROOT, "remote", "get-url", "origin"], text=True
    ).strip().rstrip("/")
    if configured_origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("El clon existente no corresponde al repositorio oficial. No se elimina.")

    # Si por error guardaste material de trabajo en el clon, PROTEGERLO.
    persistent_names = (
        "training", "source", "input", "resume_hf", "diagnostics",
        ".hf_staging", ".hf_synced_verified.json", ".studio_preferences.json",
        "dataset_validation.json",
    )
    found = [name for name in persistent_names if (project_dir / name).exists()]
    for pattern in ("*.ckpt", "dataset.jsonl"):
        if next(project_dir.rglob(pattern), None) is not None:
            found.append(pattern)
    if found:
        raise RuntimeError(
            "Hay datos potencialmente persistentes dentro del clon: "
            + ", ".join(found)
            + ". Muévelos a " + DATA_ROOT + " antes de reinstalar."
        )

    # Evitar romper un fine-tune o Gradio que esté usando este código.
    for proc in Path("/proc").iterdir():
        if not proc.name.isdecimal():
            continue
        try:
            command_line = (proc / "cmdline").read_bytes()
        except (OSError, PermissionError):
            continue
        if ROOT.encode() in command_line and (
            b"piper_train" in command_line or b"gradio_finetune" in command_line
        ):
            raise RuntimeError(
                "Hay un proceso Piper-Neo activo (PID " + proc.name
                + "). No se borra el código mientras esté en uso."
            )

# Conservar también el log antiguo, que versiones previas escribían
# dentro del clon. No sobrescribir registros ya preservados.
legacy_log = project_dir / "studio_kaggle.log"
if project_dir.exists() and legacy_log.is_file() and not legacy_log.is_symlink():
    data_dir.mkdir(parents=True, exist_ok=True)
    old_log_backup = data_dir / ("studio_kaggle.legacy-" + str(os.getpid()) + ".log")
    if old_log_backup.exists():
        import time
        old_log_backup = data_dir / ("studio_kaggle.legacy-" + str(time.time_ns()) + ".log")
    shutil.copy2(legacy_log, old_log_backup)
    print("Log anterior conservado:", old_log_backup)

# Clonar PRIMERO. Si falla Git o Internet, el checkout anterior sigue intacto.
with tempfile.TemporaryDirectory(prefix=".piper-neo-stage-", dir=str(root_parent)) as tmp:
    fresh = Path(tmp) / "Piper-Neo"
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         REPO_URL, str(fresh)],
        check=True,
    )
    if not (fresh / ".git").is_dir():
        raise RuntimeError("El nuevo clon no contiene .git. Se conserva el checkout anterior.")
    if project_dir.exists():
        shutil.rmtree(project_dir)  # Solo el código oficial verificado, nunca DATA_ROOT.
        print("Clon anterior eliminado; datos externos intactos.")
    fresh.rename(project_dir)

data_dir.mkdir(parents=True, exist_ok=True)
(data_dir / "gradio_tmp").mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(
    ["git", "-C", ROOT, "log", "-1", "--format=%h %s"], text=True
).strip()
branch = subprocess.check_output(
    ["git", "-C", ROOT, "branch", "--show-current"], text=True
).strip()
print("Clon actualizado desde GitHub:", REPO_URL)
print("Rama:", branch)
print("Último commit:", commit)
print("Datos intactos:", data_dir)


## 2. Instalar dependencias compatibles y la extensión nativa

El script oficial del proyecto instala **Python 3.10 aislado** usando `uv`, PyTorch de la serie compatible con Piper Train, Gradio, ONNX y compila `monotonic_align`. No instala datasets ni sobrescribe checkpoints. Requiere Internet en Kaggle. **Este paso no es un ejecutable precompilado**; el runtime portable es una entrega futura distinta.

In [ ]:
import subprocess, os
subprocess.run(
    ["bash", str(project_dir / "script/setup_kaggle_finetune.sh")],
    cwd=str(project_dir),
    check=True,
    env={**os.environ, "PIPER_FINETUNE_ROOT": DATA_ROOT, "GRADIO_TEMP_DIR": DATA_ROOT + "/gradio_tmp", "MPLBACKEND": "Agg", "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")},
)
PY = project_dir / "src/python/.venv/bin/python"
if not PY.is_file():
    raise RuntimeError("No se creó Python 3.10 del proyecto: " + str(PY))
print("Entorno instalado:", PY)

## 3. Confirmar GPU y hacer pruebas de calidad

Esta celda no inicia fine-tune: solo prueba imports CUDA, muestra la GPU (Tesla T4 u otra NVIDIA compatible) y corre los tests del respaldo HF, reanudación e interfaz. Si CUDA da `False`, revisa Accelerator GPU en Kaggle antes de entrenar.

In [ ]:
import subprocess, os
env = {
    **os.environ,
    "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", ""),
    "MPLBACKEND": "Agg",
}
gpu_code = """
import torch
from piper_train.gradio_finetune import build_ui
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Gradio components:", len(build_ui().blocks))
"""
subprocess.run([str(PY), "-c", gpu_code], cwd=str(project_dir / "src/python"), env=env, check=True)
subprocess.run(
    [str(PY), "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-v"],
    cwd=str(project_dir / "src/python"), env=env, check=True,
)
print("Verificaciones completadas.")

## 4. Configurar secretos (opcional, recomendado)

En **Kaggle → Add-ons → Secrets**, puedes agregar:

- `HF_TOKEN` para repositorios de Hugging Face.
- `PIPER_STUDIO_AUTH_USER` y `PIPER_STUDIO_AUTH_PASSWORD` para proteger el enlace de Gradio.

Los secretos se pasan únicamente al proceso servidor y no se imprimen ni se guardan en el notebook. Si dejas user/password sin definir, Gradio no pide contraseña.

In [ ]:
import os
from pathlib import Path

server_env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "PIPER_FINETUNE_ROOT": DATA_ROOT,
    "GRADIO_TEMP_DIR": DATA_ROOT + "/gradio_tmp",
    "MPLBACKEND": "Agg",  # Kaggle usa matplotlib_inline; entorno aislado requiere Agg.
    "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", ""),
}
try:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for key in ("HF_TOKEN", "PIPER_STUDIO_AUTH_USER", "PIPER_STUDIO_AUTH_PASSWORD"):
        try:
            value = secret_client.get_secret(key)
            if value:
                server_env[key] = value
        except Exception:
            pass  # El secreto es opcional.
except ImportError:
    print("kaggle_secrets no está disponible (fuera de Kaggle).")

print("Token HF configurado:", bool(server_env.get("HF_TOKEN")))
print("Protección Gradio configurada:", bool(
    server_env.get("PIPER_STUDIO_AUTH_USER") and
    server_env.get("PIPER_STUDIO_AUTH_PASSWORD")
))

## 5. Abrir Gradio con 5 pestañas

La celda inicia el servidor en un **proceso independiente** y muestra la URL temporal `gradio.live`. No vuelvas a ejecutarla si ya está activo. Los archivos, checkpoints y logs se guardan en `/kaggle/working/piper_finetune` (fuera del repositorio Git), no en el navegador. Kaggle puede cerrar su runtime; descarga respaldos HF regularmente.

**Pestañas:** Dataset · Preprocess · Entrenamiento/Resume HF · Estado/Respaldo incremental · Inferencia/ONNX.

In [ ]:
import subprocess, time, re
from pathlib import Path

PORT = 7860
web_log = Path(DATA_ROOT) / "studio_kaggle.log"
existing = globals().get("studio_process")
if existing is not None and existing.poll() is None:
    print("Gradio ya corre, PID", existing.pid)
else:
    handle = web_log.open("a", encoding="utf-8")
    studio_process = subprocess.Popen(
        [str(PY), "-u", "-m", "piper_train.gradio_finetune",
         "--share", "--port", str(PORT)],
        cwd=str(project_dir / "src/python"),
        env=server_env,
        stdout=handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle.close()
    print("Gradio iniciado, PID:", studio_process.pid)

url = None
for _ in range(90):
    content = web_log.read_text(encoding="utf-8", errors="replace") if web_log.exists() else ""
    matches = re.findall(r"https://[a-zA-Z0-9.-]+\.gradio\.live", content)
    if matches:
        url = matches[-1]
        break
    if studio_process.poll() is not None:
        raise RuntimeError("Gradio terminó inesperadamente:\n" + content[-4000:])
    time.sleep(1)
print("URL pública:", url or "Aún generándose: revisa studio_kaggle.log")
print("El enlace requiere contraseña solamente si configuraste ambos secretos de acceso.")

## 6. Revisar servidor sin reiniciar el entrenamiento

Puedes ejecutar esta celda cuando quieras. Si Gradio sigue corriendo, recarga su URL. Si debes reconstruir el runtime, usa `Resume de corrida`, prepara el mismo dataset y usa **Hugging Face: último válido** para descargar el checkpoint y recuperar época/optimizadores.

In [ ]:
from pathlib import Path
status = globals().get("studio_process")
print("Servidor:", "activo" if status and status.poll() is None else "no activo en este kernel")
if web_log.exists():
    print("\n".join(web_log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
print("Datos persistentes:", Path(DATA_ROOT).exists())